<Info>
**Prerequisites.** If you are unfamiliar with ODEs or the Euler integration step, first read [ODEs and the Euler method](/aiml-common/lectures/ml-math/calculus/odes-and-euler/index). The discrete update in that section is also used here to track particles through the storm field.
</Info>

A storm is a familiar example of a velocity field. Its wind has three properties:

- Wind has direction.
- Wind has magnitude.
- Different locations experience different winds.

Mathematically, the wind in a storm forms a velocity field:

$$
\mathbf{v}(x, y, t)
$$

At each location $(x, y)$ and time $t$, the vector gives the direction and speed of the air.

## A rotating storm

A simple 2D rotating storm centered at the origin is

$$
\mathbf{v}(x, y) =
\begin{bmatrix} -y \\ x \end{bmatrix}.
$$

This field produces pure rotation. Its values at two points are:

- At $(1, 0)$, $\mathbf{v} = (0, 1)$, so the wind blows upward.
- At $(0, 1)$, $\mathbf{v} = (-1, 0)$, so the wind blows to the left.

The field therefore rotates counterclockwise around the origin.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

x = np.linspace(-5, 5, 20)
y = np.linspace(-5, 5, 20)
X, Y = np.meshgrid(x, y)

U = -Y
V = X

fig, ax = plt.subplots(figsize=(6, 6))
ax.quiver(X, Y, U, V, color="C0")
ax.set_title("Pure rotation: $\\mathbf{v}(x, y) = (-y, x)$")
ax.set_xlabel("x"); ax.set_ylabel("y")
ax.set_aspect("equal")
ax.grid(alpha=0.3)
plt.show()

## Velocity weakens with distance

A real storm is strongest near the eye and weakens outward. A simple model for this behavior is

$$
\mathbf{v}(x, y) =
\frac{1}{x^2 + y^2 + \varepsilon}
\begin{bmatrix} -y \\ x \end{bmatrix},
$$

where $\varepsilon > 0$ avoids the singularity at the origin. The field is strong near the center and decays with distance. The following plot shows this behavior.

In [ ]:
eps = 0.5
norm = X**2 + Y**2 + eps
U = -Y / norm
V = X / norm

fig, ax = plt.subplots(figsize=(6, 6))
ax.quiver(X, Y, U, V, color="C2")
ax.set_title("Rotation with distance falloff")
ax.set_xlabel("x"); ax.set_ylabel("y")
ax.set_aspect("equal")
ax.grid(alpha=0.3)
plt.show()

## Particles in the storm

Particle motion in a velocity field is a dynamical system. A particle at position $\mathbf{x}(t)$ moves according to

$$
\frac{d \mathbf{x}}{d t} = \mathbf{v}(\mathbf{x}).
$$

You can integrate this numerically. The simplest scheme is the Euler step,

$$
\mathbf{x}(t + \Delta t) = \mathbf{x}(t) + \Delta t \cdot \mathbf{v}(\mathbf{x}(t)),
$$

Repeat this update to follow many particles released at $t = 0$. The resulting trajectories model the motion of raindrops, leaves, and debris in a storm.

In [ ]:
def storm_v(p, eps=0.5):
    x, y = p[..., 0], p[..., 1]
    norm = x**2 + y**2 + eps
    return np.stack([-y / norm, x / norm], axis=-1)


rng = np.random.default_rng(0)
n_particles = 60
n_steps = 400
dt = 0.05

particles = rng.uniform(-4, 4, size=(n_particles, 2))
trajectories = np.empty((n_steps + 1, n_particles, 2))
trajectories[0] = particles

for step in range(n_steps):
    trajectories[step + 1] = trajectories[step] + dt * storm_v(trajectories[step])

In [ ]:
fig, ax = plt.subplots(figsize=(7, 7))
colors = plt.cm.viridis(np.linspace(0, 1, n_particles))
for i in range(n_particles):
    ax.plot(trajectories[:, i, 0], trajectories[:, i, 1],
            color=colors[i], alpha=0.6, lw=0.8)
    ax.scatter(trajectories[-1, i, 0], trajectories[-1, i, 1],
               color=colors[i], s=14, zorder=3, edgecolor="white", linewidth=0.5)
ax.set_title(f"{n_particles} particles integrated through the storm field")
ax.set_xlabel("x"); ax.set_ylabel("y")
ax.set_xlim(-5, 5); ax.set_ylim(-5, 5)
ax.set_aspect("equal")
ax.grid(alpha=0.3)
plt.show()

## Divergence: rotation against contraction

Rotation moves particles without changing how crowded they are. A cloud of particles with a rotationally symmetric density looks the same after the storm has turned it. To see a field that does change the density, set a contracting field beside the rotation:

$$
\mathbf{v}_{\text{rot}}(x, y) = \begin{bmatrix} -y \\ x \end{bmatrix},
\qquad
\mathbf{v}_{\text{con}}(x, y) = \begin{bmatrix} -x \\ -y \end{bmatrix}.
$$

The contracting field points at the origin everywhere, with speed proportional to the distance from it. The divergence measures how much a field spreads particles apart at a point:

$$
\nabla \cdot \mathbf{v} = \frac{\partial v_x}{\partial x} + \frac{\partial v_y}{\partial y}.
$$

For the rotation, $\nabla \cdot \mathbf{v}_{\text{rot}} = 0 + 0 = 0$. It preserves area, so a rotationally symmetric density does not change. For the contraction, $\nabla \cdot \mathbf{v}_{\text{con}} = -1 - 1 = -2$. Particles crowd together and the density rises.

Both fields are linear, $\mathbf{v} = A\mathbf{x}$, so the flow map over a time $t$ is the matrix $e^{tA}$. Its determinant is $e^{t \operatorname{tr} A} = e^{t \nabla \cdot \mathbf{v}}$. The rotation has determinant $1$ and the contraction has determinant $e^{-2t}$. The code below moves a cloud of particles drawn from a standard Gaussian with the Euler step from the section on particles in the storm.

In [ ]:
A_rot = np.array([[0.0, -1.0], [1.0, 0.0]])
A_con = np.array([[-1.0, 0.0], [0.0, -1.0]])

T = 1.0
n_euler = 1000
h = T / n_euler


def flow(points, A):
    """Move points through the linear field v = A x for time T with Euler steps."""
    p = points.copy()
    for _ in range(n_euler):
        p = p + h * p @ A.T
    return p


rng = np.random.default_rng(1)
cloud = rng.standard_normal((20000, 2))
cloud_rot = flow(cloud, A_rot)
cloud_con = flow(cloud, A_con)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 5), sharex=True, sharey=True)
panels = [(cloud_rot, "Rotation, divergence 0"), (cloud_con, "Contraction, divergence -2")]
for ax, (pts, title) in zip(axes, panels):
    ax.hist2d(pts[:, 0], pts[:, 1], bins=60, range=[[-3, 3], [-3, 3]], cmap="viridis")
    ax.set_title(title)
    ax.set_xlabel("x")
    ax.set_aspect("equal")
axes[0].set_ylabel("y")
plt.show()


Both panels start from the same Gaussian cloud and run for time $T = 1$. The left panel shows the cloud after the rotation: a round Gaussian blob, because turning a rotationally symmetric cloud leaves it unchanged. The right panel shows the cloud after the contraction: it is concentrated near the origin, with a higher peak.

The area claim can be measured directly. Follow the corners of a small square and measure the polygon area with the shoelace formula. The rotation keeps the area and the contraction shrinks it by $e^{-2T}$.

In [ ]:
def polygon_area(poly):
    x, y = poly[:, 0], poly[:, 1]
    return 0.5 * abs(np.dot(x, np.roll(y, -1)) - np.dot(y, np.roll(x, -1)))


square = np.array([[1.0, 1.0], [1.2, 1.0], [1.2, 1.2], [1.0, 1.2]])
area0 = polygon_area(square)
area_rot = polygon_area(flow(square, A_rot))
area_con = polygon_area(flow(square, A_con))

print(f"initial area          {area0:.5f}")
print(f"after rotation        {area_rot:.5f}")
print(f"after contraction     {area_con:.5f}   ratio {area_con / area0:.4f}, exp(-2T) = {np.exp(-2 * T):.4f}")

assert np.isclose(area_rot / area0, 1.0, rtol=2e-3)
assert np.isclose(area_con / area0, np.exp(-2 * T), rtol=2e-3)


The same factor shows up in the Gaussian. A contraction maps $\mathcal{N}(0, \Sigma)$ to $\mathcal{N}(0, e^{-2t}\Sigma)$, so the covariance of the cloud shrinks by $e^{-2t}$. The rotation maps $\mathcal{N}(0, I)$ to itself.

In [ ]:
cov0 = np.cov(cloud.T)
cov_rot = np.cov(cloud_rot.T)
cov_con = np.cov(cloud_con.T)

print("covariance after rotation:\n", np.round(cov_rot, 3))
print("covariance after contraction:\n", np.round(cov_con, 3))
print(f"trace ratio under contraction {np.trace(cov_con) / np.trace(cov0):.4f}, exp(-2T) = {np.exp(-2 * T):.4f}")

assert np.allclose(cov_rot, np.eye(2), atol=5e-2)
assert np.allclose(cov_con, np.exp(-2 * T) * np.eye(2), atol=1e-2)


Divergence alone does not give the new density. The [dynamic transport section](/aiml-common/lectures/optimal-transport/dynamic-transport/index) derives the continuity equation, where the divergence of $p\,\mathbf{v}$ is what changes the density.